# Episode-aware power-grid cyberattack detection

**Research question.** Can causal temporal features and simplified electrical-consistency residuals distinguish normal telemetry from FDI, DoS, and replay events in previously unseen operating episodes?

This notebook is a readable companion to the scripted experiment. The authoritative reproducible run is `python src/train_cyberattack_detector.py`; the notebook deliberately reuses the same package functions rather than maintaining a second implementation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib
if 'ipykernel' not in sys.modules:
    matplotlib.use('Agg')  # safe when the code cells are smoke-tested headlessly
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from powergrid_security.features import build_features
from powergrid_security.simulation import SimulationConfig, generate_dataset

DATA_PATH = PROJECT_ROOT / 'data' / 'synthetic_grid_cyberattack_data.csv'
data = pd.read_csv(DATA_PATH) if DATA_PATH.exists() else generate_dataset(SimulationConfig())
data.shape

## 1. Inspect the independent units

Rows within an episode are autocorrelated. Therefore, episodes—not rows—are the sampling and splitting unit. Attacked episodes include a normal pre-event period, producing realistic class imbalance.

In [ ]:
display(data.groupby('scenario')['episode_id'].nunique().rename('episodes').to_frame())
display(data['attack_type'].value_counts().rename('rows').to_frame())
assert not data.duplicated(['episode_id', 'timestep']).any()

In [ ]:
example_id = data.loc[data['scenario'] == 'false_data_injection', 'episode_id'].iloc[0]
example = data[data['episode_id'] == example_id]
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(example['timestep'], example['load_mw'], label='Load (MW)')
axes[1].plot(example['timestep'], example['voltage_pu'], label='Voltage (p.u.)', color='tab:orange')
attack = example['attack_type'] != 'normal'
for ax in axes:
    ax.fill_between(example['timestep'], 0, 1, where=attack, transform=ax.get_xaxis_transform(), alpha=.15, color='red', label='Attack window')
    ax.legend(loc='best')
axes[1].set_xlabel('15-minute timestep')
fig.suptitle(f'Example held sequence: {example_id}')
fig.tight_layout()

## 2. Build causal features

Features use the current and trailing observations within each episode. The target, scenario, event identifier, severity, and episode identifier are excluded. The test suite verifies that changing a future observation cannot change an earlier feature row.

In [ ]:
features = build_features(data, feature_set='full')
display(features.head())
print(f'{features.shape[1]} model features')
assert {'attack_type', 'scenario', 'event_id', 'episode_id'}.isdisjoint(features.columns)

## 3. Read the locked-holdout evidence

The script selects a model only from development-fold macro F1, then evaluates the locked episode holdout once. It also writes grouped-bootstrap intervals, per-class results, ablations, calibration diagnostics, robustness tests, and row-level predictions.

In [ ]:
comparison = pd.read_csv(PROJECT_ROOT / 'reports' / 'model_comparison.csv')
ablations = pd.read_csv(PROJECT_ROOT / 'reports' / 'ablation_study.csv')
robustness = pd.read_csv(PROJECT_ROOT / 'reports' / 'robustness_study.csv')
display(comparison.sort_values('cv_macro_f1_mean', ascending=False))
display(ablations[['feature_set', 'macro_f1', 'balanced_accuracy']])
display(robustness[['condition', 'macro_f1', 'expected_calibration_error']])

## Interpretation boundary

The study demonstrates a rigorous *workflow* on synthetic telemetry. It does not establish utility-scale detection performance. The next study should freeze this pipeline and validate it on a topology-aware, hardware-in-the-loop, PMU, or SCADA benchmark. See `DATA_CARD.md`, `MODEL_CARD.md`, and `reports/research_report.md`.